# Phase 2: Contextual Embeddings with BERT

**Goal:** See how BERT generates *different* vectors for the same word in different contexts.

**What you'll learn:**
- How to load and use BERT from HuggingFace
- How tokenization works (including special [CLS] and [SEP] tokens)
- How to extract embeddings from BERT's hidden states
- Why 'bank' in 'river bank' has a different vector than 'bank' in 'bank account'
- The difference between [CLS] pooling and mean pooling

## Step 1: Install & Import

In [5]:
# Install (run once)
# !pip install transformers torch

In [6]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from transformers import BertTokenizer, BertModel
from scipy.spatial.distance import cosine

print(f'PyTorch version: {torch.__version__}')

# Auto-detect best available device: CUDA (NVIDIA) → MPS (Apple Silicon) → CPU
if torch.cuda.is_available():
    device = 'cuda'
    print(f'Using device: CUDA (NVIDIA GPU) — {torch.cuda.get_device_name(0)}')
elif torch.backends.mps.is_available():
    device = 'mps'
    print(f'Using device: MPS (Apple Silicon GPU)')
else:
    device = 'cpu'
    print(f'Using device: CPU')

PyTorch version: 2.11.0
Using device: MPS (Apple Silicon GPU)


## Step 2: Load BERT Tokenizer and Model

We use `bert-base-uncased` — 12 transformer layers, 768 hidden dimensions, 110M parameters.
This will download ~440MB on first run (cached after that).

In [ ]:
model_name = 'bert-base-uncased'
print(f'Loading tokenizer: {model_name}...')
tokenizer = BertTokenizer.from_pretrained(model_name)

print(f'Loading model...')
model = BertModel.from_pretrained(model_name)
model.eval()  # Set to eval mode (disables dropout)
model = model.to(device)

print(f'Model loaded! Parameters: {sum(p.numel() for p in model.parameters()):,}')

Loading tokenizer: bert-base-uncased...


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Loading model...


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

## Step 3: Understand Tokenization

BERT doesn't work with raw words — it uses a **WordPiece tokenizer** that splits unknown or rare words into subword pieces. Let's see how it works.

In [ ]:
sentences = [
    "I sat by the river bank",
    "I deposited money at the bank",
]

for sent in sentences:
    tokens = tokenizer.tokenize(sent)
    token_ids = tokenizer.encode(sent)
    decoded = tokenizer.convert_ids_to_tokens(token_ids)
    
    print(f'Sentence: {sent}')
    print(f'  Tokens:    {tokens}')
    print(f'  With [CLS]/[SEP]: {decoded}')
    print(f'  Token IDs: {token_ids}')
    print()

print('Note:')
print('  [CLS] = Classification token — added at the START of every input')
print('  [SEP] = Separator token — added at the END of every input')
print('  ## prefix = subword continuation (e.g., ##ing is the "-ing" part of a word)')

## Step 4: Extract BERT Embeddings

We'll write a helper function that:
1. Tokenizes the sentence
2. Runs it through BERT
3. Returns the full last hidden state (one vector per token)

In [ ]:
def get_bert_output(sentence):
    """
    Returns:
      tokens: list of token strings
      last_hidden: tensor of shape [num_tokens, 768]
      cls_embedding: the [CLS] token vector (shape [768])
      mean_embedding: mean of all token vectors (shape [768])
    """
    # Tokenize
    inputs = tokenizer(sentence, return_tensors='pt').to(device)
    tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])

    # Run through BERT (no gradient needed for inference)
    with torch.no_grad():
        outputs = model(**inputs)

    # last_hidden_state shape: [batch=1, seq_len, hidden=768]
    last_hidden = outputs.last_hidden_state[0]  # Remove batch dim -> [seq_len, 768]

    cls_embedding  = last_hidden[0]             # [CLS] is always the first token
    mean_embedding = last_hidden.mean(dim=0)    # Average all token vectors

    return tokens, last_hidden, cls_embedding, mean_embedding

# Test it
tokens, hidden, cls_emb, mean_emb = get_bert_output("Hello world")
print(f'Tokens: {tokens}')
print(f'Hidden state shape: {hidden.shape}  (tokens × hidden_dim)')
print(f'[CLS] embedding shape: {cls_emb.shape}')
print(f'Mean embedding shape: {mean_emb.shape}')

## Step 5: The Key Demo — Same Word, Different Vectors

This is the core difference between Word2Vec and BERT.
Let's extract the embedding for the word 'bank' in two very different contexts.

In [ ]:
def cosine_similarity(v1, v2):
    """Compute cosine similarity between two vectors."""
    v1 = v1.cpu().numpy()
    v2 = v2.cpu().numpy()
    return 1 - cosine(v1, v2)  # scipy cosine returns distance, so 1 - distance = similarity

def get_word_embedding(sentence, target_word):
    """Get the embedding for a specific word in a sentence."""
    tokens, hidden, _, _ = get_bert_output(sentence)
    
    # Find the index of the target word in tokens
    # (target might be split into subwords — we take the first subword piece)
    target_idx = None
    for i, token in enumerate(tokens):
        if token.lower() == target_word.lower():
            target_idx = i
            break
    
    if target_idx is None:
        print(f'Warning: "{target_word}" not found as a token. Tokens: {tokens}')
        return None
    
    return hidden[target_idx]


# The two sentences
sent_river  = "I sat by the river bank yesterday"
sent_finance = "I deposited all my money at the bank"

bank_river   = get_word_embedding(sent_river, 'bank')
bank_finance = get_word_embedding(sent_finance, 'bank')

sim = cosine_similarity(bank_river, bank_finance)

print('BERT Contextual Embedding Demo')
print('=' * 50)
print(f'Sentence A: "{sent_river}"')
print(f'Sentence B: "{sent_finance}"')
print()
print(f'Cosine similarity of "bank" between A and B: {sim:.4f}')
print()
print('Interpretation:')
if sim < 0.85:
    print(f'  Score {sim:.4f} < 0.85: BERT correctly produces DIFFERENT vectors for each sense!')
else:
    print(f'  Score {sim:.4f}: Vectors are similar (small model, consider using larger BERT variant)')

## Step 6: Compare Multiple Contexts Side by Side

In [ ]:
# Let's run a broader comparison: word 'right' has many meanings
right_contexts = [
    ("Turn right at the next intersection", "right"),          # direction
    ("You have the right to remain silent", "right"),          # legal right
    ("That answer is right", "right"),                         # correct
    ("The right wing of the building is closed", "right"),     # side/wing
]

print('Cosine Similarity Matrix for "right" in different contexts:')
print()

# Get all embeddings
embeddings = []
labels = []
for sent, word in right_contexts:
    emb = get_word_embedding(sent, word)
    if emb is not None:
        embeddings.append(emb)
        labels.append(sent[:40] + '...')

# Build similarity matrix
n = len(embeddings)
sim_matrix = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        sim_matrix[i][j] = cosine_similarity(embeddings[i], embeddings[j])

# Display
fig, ax = plt.subplots(figsize=(10, 4))
im = ax.imshow(sim_matrix, cmap='YlOrRd', vmin=0.7, vmax=1.0)
ax.set_xticks(range(n))
ax.set_yticks(range(n))
ax.set_xticklabels([f'Ctx {i+1}' for i in range(n)], rotation=15)
ax.set_yticklabels([f'Ctx {i+1}: {l}' for i, l in enumerate(labels)], fontsize=9)
plt.colorbar(im, ax=ax)
ax.set_title('Similarity of "right" across different contexts\n(lower = more different meanings)', fontsize=12)

for i in range(n):
    for j in range(n):
        ax.text(j, i, f'{sim_matrix[i][j]:.2f}', ha='center', va='center', fontsize=10)

plt.tight_layout()
plt.savefig('phase2_context_similarity.png', dpi=150)
plt.show()

## Step 7: [CLS] vs Mean Pooling for Sentence Embeddings

In [ ]:
test_sentences = [
    "The cat sat on the mat",
    "A kitten rested on the rug",       # semantically similar to above
    "The stock market crashed today",   # unrelated
]

cls_embeddings  = []
mean_embeddings = []

for sent in test_sentences:
    _, _, cls_emb, mean_emb = get_bert_output(sent)
    cls_embeddings.append(cls_emb)
    mean_embeddings.append(mean_emb)

print('Sentence similarity comparison:')
print(f'  S1: "{test_sentences[0]}"')
print(f'  S2: "{test_sentences[1]}"  (should be SIMILAR to S1)')
print(f'  S3: "{test_sentences[2]}"  (should be DIFFERENT from S1)')
print()

for method, embeds in [('[CLS] pooling', cls_embeddings), ('Mean pooling', mean_embeddings)]:
    s1_s2 = cosine_similarity(embeds[0], embeds[1])
    s1_s3 = cosine_similarity(embeds[0], embeds[2])
    print(f'{method}:')
    print(f'  S1 ↔ S2 (similar): {s1_s2:.4f}')
    print(f'  S1 ↔ S3 (different): {s1_s3:.4f}')
    print()

print('Takeaway: Mean pooling often gives better sentence-level representations than [CLS].')
print('For best results, Phase 3 uses SBERT — which is BERT specifically fine-tuned for this!')

## Step 8: Visualize How Layers Build Up Understanding

BERT has 12 layers. Early layers capture syntax, later layers capture semantics.
Let's see how the representation of 'bank' changes across layers.

In [ ]:
def get_all_layer_embeddings(sentence, target_word):
    """Get the embedding for target_word at every BERT layer."""
    inputs = tokenizer(sentence, return_tensors='pt').to(device)
    tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])

    # output_hidden_states=True returns all 13 hidden states (embedding + 12 layers)
    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)

    target_idx = next((i for i, t in enumerate(tokens) if t.lower() == target_word.lower()), None)
    if target_idx is None:
        return None, None

    # hidden_states is a tuple of 13 tensors, each [batch, seq, 768]
    layer_embeddings = [hs[0, target_idx].cpu().numpy() for hs in outputs.hidden_states]
    return layer_embeddings, tokens

# Get layer-by-layer embeddings for 'bank' in both contexts
layers_river,   _ = get_all_layer_embeddings(sent_river, 'bank')
layers_finance, _ = get_all_layer_embeddings(sent_finance, 'bank')

# Compute cosine similarity at each layer
layer_sims = []
for l_r, l_f in zip(layers_river, layers_finance):
    sim = 1 - cosine(l_r, l_f)
    layer_sims.append(sim)

# Plot
plt.figure(figsize=(10, 5))
plt.plot(range(13), layer_sims, 'o-', color='royalblue', linewidth=2, markersize=8)
plt.axhline(y=layer_sims[0], color='gray', linestyle='--', alpha=0.5, label='Layer 0 (input embedding)')
plt.xlabel('BERT Layer (0 = input embedding, 12 = final layer)', fontsize=12)
plt.ylabel('Cosine Similarity', fontsize=12)
plt.title('How "bank" diverges across BERT layers\n"river bank" vs "bank account"\n(lower = model has learned they are different)', fontsize=12)
plt.xticks(range(13))
plt.ylim(0.5, 1.05)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig('phase2_layer_divergence.png', dpi=150)
plt.show()

print('As we go deeper in BERT, the two senses of "bank" diverge more.')
print('Early layers handle syntax; later layers encode meaning.')

## Transformer Architecture — Deep Dive

Everything in Phase 2 builds on the transformer architecture. This section captures the key concepts from first principles so you can reason about what BERT is actually doing at every step.

---

### 1. Evolution to BERT — Why Was It Needed?

Before BERT, language models processed text sequentially using RNNs and LSTMs. Each step depended on the previous one:

```
RNN / LSTM (pre-2017):
  "I" → hidden state h1
          ↓
  "sat" → hidden state h2  (uses h1)
             ↓
  "by" → hidden state h3   (uses h2)
            ↓
  "the" → h4  →  "bank" → h5

Problem 1: Sequential — can't parallelize, very slow to train on large data
Problem 2: Long-range forgetting — h5 barely remembers "I" from the start
Problem 3: One direction only — "bank" only sees words to its LEFT
```

**2017 — Attention Is All You Need (Vaswani et al.)**

Replaced recurrence entirely with self-attention. Every token attends to every other token simultaneously — no sequential dependency. This enabled massive parallelization on GPUs and is why modern models can train on billions of sentences.

```
Transformer Encoder (2017):
  All tokens processed simultaneously
  Every token attends to every other token in both directions
  No recurrence — pure matrix operations
```

**2018 — BERT (Bidirectional Encoder Representations from Transformers)**

GPT (also 2018) used the transformer decoder — left-to-right only. BERT's innovation was using the encoder with a masked training objective, forcing truly bidirectional understanding:

```
GPT  →  left-to-right only  →  "bank" sees [I, sat, by, the, river] only
BERT →  bidirectional        →  "bank" sees [I, sat, by, the, river] AND everything after
```

The masked language modeling task (predict [MASK] from both directions) is what made this possible — you cannot cheat and predict a masked word by only looking left to right.

---

### 2. Query, Key, Value — The Core Mechanism

Every token simultaneously plays three roles in self-attention. The best intuition is a **library search**:

```
Q (Query)  →  what am I looking for?
               "I am 'bank' — I'm looking for location or finance context clues"

K (Key)    →  what do I contain / offer?
               "I am 'river' — I offer geographical/water context"
               "I am 'money' — I offer financial context"

V (Value)  →  what information do I actually pass forward as my contribution?
               the actual payload that gets blended into the result
```

The precise formula:

```
Attention(Q, K, V) = softmax( Q × Kᵀ / √dk ) × V
                         ─────────────────────
                              attention weights
                         (how much of each V to collect)
```

**Q × Kᵀ** — relevance scores: how well does this token's query match each other token's key?
**÷ √dk** — scale by √64 = 8 to prevent scores from getting too large and softmax becoming too sharp
**softmax** — converts raw scores to probabilities that sum to 1.0
**× V** — collect actual information from each token, weighted by those probabilities

**The key insight:** Q and K are only the matching mechanism — they decide WHO to attend to. V is the actual payload — the information that gets blended into the output. The output for "bank" is a weighted blend of every token's V vector, where the weights were determined by how well bank's Q matched each token's K.

---

### 3. Full Picture — One Token Through One Block

Here is the complete transformation a single token goes through inside one transformer block:

```
"bank" embedding [768d]  (output from previous block or embedding layer)
        ↓
┌──────────────────────────── Multi-Head Attention ──────────────────────────────┐
│                                                                                 │
│  Step 1 — Project ALL tokens into Q, K, V simultaneously:                      │
│    Q = X × W_Q   (5×768) × (768×64) = (5×64)  ← all 5 token queries at once  │
│    K = X × W_K   (5×768) × (768×64) = (5×64)  ← all 5 token keys at once     │
│    V = X × W_V   (5×768) × (768×64) = (5×64)  ← all 5 token values at once   │
│                                                                                 │
│  Step 2 — Attention scores (bank's Q vs every token's K):                      │
│    scores = Q × Kᵀ   (5×64) × (64×5) = (5×5) ← full attention matrix         │
│                                                                                 │
│  Step 3 — Scale + Softmax (row-wise, each row sums to 1.0):                   │
│    weights = softmax(scores / √64)                                              │
│    bank row: [I=0.18, sat=0.12, by=0.35, the=0.10, bank=0.25]                 │
│                                                                                 │
│  Step 4 — Weighted sum of Values:                                               │
│    bank output = 0.18×I_V + 0.12×sat_V + 0.35×by_V + 0.10×the_V + 0.25×bank_V│
│    → new 64-dim vector (context-aware)                                          │
│                                                                                 │
│  This runs 12 times in parallel (12 heads, each with own W_Q, W_K, W_V):      │
│  12 × 64d outputs → concatenate → 768d → multiply by W_O → 768d               │
└─────────────────────────────────────────────────────────────────────────────────┘
        ↓
   Add residual + LayerNorm   (input + attention output, then normalize)
        ↓
┌──────────────── Feed Forward Network ──────────────────┐
│  Linear(768 → 3072) → GELU activation → Linear(3072 → 768) │
│  Runs independently per token — no cross-token mixing here  │
└──────────────────────────────────────────────────────────────┘
        ↓
   Add residual + LayerNorm
        ↓
"bank" refined vector [768d]  → input to next Block
```

---

### 4. How All 12 Heads and 12 Blocks Relate

Two levels of structure — heads and blocks — are often confused:

```
Input sentence tokens [768d each]
        ↓
┌─────────────────────────────────────────┐
│              Block 1                    │
│  ┌──────┬──────┬──────┬─────┬──────┐   │
│  │Head 1│Head 2│Head 3│ ... │Head12│   │  ← 12 heads PARALLEL
│  │W_Q1  │W_Q2  │W_Q3  │     │W_Q12 │   │    each learns different
│  │W_K1  │W_K2  │W_K3  │     │W_K12 │   │    relationship type
│  │W_V1  │W_V2  │W_V3  │     │W_V12 │   │
│  └──────┴──────┴──────┴─────┴──────┘   │
│  concatenate 12×64=768 → W_O → FFN     │
└─────────────────────────────────────────┘
        ↓  output of Block 1 feeds into Block 2
┌─────────────────────────────────────────┐
│              Block 2                    │
│  ┌──────┬──────┬──────┬─────┬──────┐   │
│  │Head 1│Head 2│Head 3│ ... │Head12│   │  ← 12 heads PARALLEL
│  └──────┴──────┴──────┴─────┴──────┘   │
│  concatenate + FFN                      │
└─────────────────────────────────────────┘
        ↓
       ...
        ↓
┌─────────────────────────────────────────┐
│              Block 12                   │
│  ┌──────┬──────┬──────┬─────┬──────┐   │
│  │Head 1│Head 2│Head 3│ ... │Head12│   │  ← 12 heads PARALLEL
│  └──────┴──────┴──────┴─────┴──────┘   │
│  concatenate + FFN                      │
└─────────────────────────────────────────┘
        ↓
Final output: one context-aware 768d vector per token
```

**Parallelism has three levels:**

| Level | Parallel? | Why |
|---|---|---|
| Tokens within a head | Yes | Pure matrix multiply — no token depends on another |
| Heads within a block | Yes | Each head has independent W_Q, W_K, W_V |
| Blocks 1 through 12 | No | Block N needs Block N-1's output to refine meaning |

What each block progressively learns:
```
Block 1–3  →  surface patterns — which words are near each other
Block 4–6  →  phrases and syntactic structure
Block 7–9  →  semantic context — what domain is this?
Block 10–12 →  precise meaning of each token in full sentence context
```

Each head in each block specializes in a different relationship — subject-verb agreement, pronoun reference, semantic similarity — no one programs this, it emerges from training.

---

### 5. Pre-training vs Inference — The Critical Distinction

**Pre-training (done once by the research team on thousands of TPUs):**

```
Input:   "I sat by the [MASK] bank"
Target:  predict "river" at the [MASK] position

Step 1 — Forward pass (same computation as inference):
  Tokenize → embed → 12 blocks → output vector for [MASK] position
  Predict over 30,522 vocab: river=0.12, money=0.08 ...  ← wrong at start

Step 2 — Compute loss:
  Correct answer is "river" (should have probability 1.0)
  Loss = -log(0.12) = 2.12  ← high, model needs to improve

Step 3 — Backpropagate:
  Gradient flows BACKWARD: Block 12 → Block 11 → ... → Block 1
  Every W_Q, W_K, W_V, W_FFN in all 12 blocks nudged slightly
  "river" and "bank" learn to attend to each other more strongly

Step 4 — Repeat billions of times across billions of sentences
  Weights converge → saved as pytorch_model.bin (109M frozen parameters)
```

**Inference (what Phase 2 does — weights are frozen, no learning):**

```
Input:   "I sat by the river bank"
Goal:    get context-aware vector for "bank"

Step 1 — Tokenize:
  [CLS] i sat by the river bank [SEP]  →  [101, 1045, 2938, 2011, 1996, 2314, 2924, 102]

Step 2 — Look up base embeddings (static, same every time):
  token_embed + position_embed + segment_embed = 768d input per token
  "bank" base vector: [0.34, -0.12, 0.78, ...]  ← always identical starting point

Step 3 — Block 1 (frozen weights):
  All Q, K, V computed for all 8 tokens simultaneously
  "bank" attends to "river" with high weight (learned during pre-training)
  "bank" vector updated — first hint of river/geography context

Step 4 — Blocks 2–11 (frozen weights):
  Each block refines further — "river" influence on "bank" grows stronger

Step 5 — Block 12 (frozen weights):
  Final "bank" vector: [0.67, -0.34, 0.56, ...]  ← geography meaning

No gradients. No weight updates. Pure forward pass.
```

**The contrast:**

```
Pre-training:  forward pass + loss + backprop  →  weights CHANGE  (offline, by the research team)
Inference:     forward pass only               →  weights FROZEN  (what you run in Phase 2)
Fine-tuning:   forward pass + small loss       →  weights change slightly (Phase 5)
```

---

### 6. Why "bank" Gets Two Different Vectors

Same token ID (2924), same base embedding, completely different final vectors:

```
Sentence A: "I sat by the river bank"
  Base "bank": [0.34, -0.12, 0.78, ...]
  Block 1: "bank" attends heavily to "river" → vector shifts toward geography
  Block 12: [0.67, -0.34, 0.56, ...]  ← river bank meaning

Sentence B: "I deposited money at the bank"
  Base "bank": [0.34, -0.12, 0.78, ...]  ← identical start
  Block 1: "bank" attends heavily to "deposited", "money" → different shift
  Block 12: [0.12,  0.89, -0.23, ...]  ← financial institution meaning
```

There are never two stored vectors for "bank" in BERT. One base embedding + two different computations = two different contextual vectors. This is the fundamental difference from Word2Vec which stored one fixed vector per word with no mechanism to change it at inference time.

---

## Key Takeaways

### 1. WordPiece Tokenization — Not Words, Sub-Word Pieces
BERT never works with raw words. Every sentence is split into sub-word tokens by WordPiece, then padded with special tokens: **[CLS]** at the start (sentence-level summary), **[SEP]** at the end (sentence boundary), and **[MASK]** during pre-training (hidden tokens for the model to predict). Rare or unknown words ("unembedding", "GPT-4") are split into known pieces ("un", "##em", "##bed", ...) so the vocabulary stays fixed at 30,522 tokens. This is why BERT can handle any word even if it was never seen during training.

### 2. Contextual Embeddings — Same Word, Different Vectors
The defining feature of BERT vs Word2Vec. Word2Vec stores one fixed vector per word — "bank" always maps to the same point, regardless of whether you mean a riverbank or a financial institution. BERT starts with the same base embedding for both uses of "bank", but the Q/K/V attention mechanism in each of the 12 blocks re-weights that vector based on surrounding tokens. After 12 blocks, "bank" next to "river" has drifted to a completely different point in 768-dimensional space than "bank" next to "money". No separate vectors are stored — context is computed fresh each time.

### 3. Q/K/V Attention — Each Token Asks, Answers, and Contributes
For every token in the sentence, BERT computes three vectors using learned weight matrices: **Query** ("what am I looking for?"), **Key** ("what do I contain?"), **Value** ("what do I contribute if selected?"). Attention scores are softmax(QKt / sqrt(64)) — the dot product measures alignment between a token's query and every other token's key. The result is a weighted sum of values: tokens that are semantically relevant to each other transfer their value information. This is why "bank" near "river" gets high attention weight toward "river" — their Q x K dot product is high.

### 4. Multi-Head Attention — 12 Parallel Perspectives Per Block
Each of BERT's 12 transformer blocks runs 12 attention heads **in parallel**, not sequentially. Each head gets its own 64-dimensional projection (12 x 64 = 768). Different heads learn to attend to different relationships — one head might specialize in subject-verb agreement, another in coreference, another in positional proximity. Their outputs are concatenated back to 768d, then projected through a Feed-Forward Network (768 -> 3072 -> 768 with GELU activation). Running 12 heads simultaneously is what makes BERT capture many types of linguistic structure at once.

### 5. 12 Blocks Are Sequential — Each Refines the Previous Layer
The 12 transformer blocks run **one after another**, each block receiving the output of the previous block as input. Early blocks (1-4) tend to capture low-level syntax: word order, part-of-speech, punctuation structure. Middle blocks (5-8) capture phrase-level patterns: noun phrases, verb phrases, clausal structure. Later blocks (9-12) capture semantic meaning: word sense disambiguation, entity types, sentence-level reasoning. By block 12, each token's 768-dimensional vector encodes everything BERT knows about its role in that specific sentence. Residual connections (adding the input back to the output of each sub-layer) ensure earlier features are never completely forgotten.

### 6. Pre-training vs Inference — Weights Change Only During Training
BERT's 110M parameters were trained over weeks on thousands of TPUs using Masked Language Modeling (predict [MASK] tokens) and Next Sentence Prediction (does sentence B follow sentence A?). During training, every forward pass computes a loss, and backpropagation nudges all 110M weights. Once training finishes, the weights are **frozen**. In Phase 2, when you call model(**inputs), you are running pure forward pass: matrix multiplications, no gradient computation, no weight updates. The frozen Q/K/V weight matrices are what give BERT its learned intuitions about language — you are using the published pre-trained weights, not re-doing the training.

### 7. [CLS] vs Mean Pooling — Two Ways to Get a Sentence Vector
BERT outputs one 768-dimensional vector **per token**, not one per sentence. To get a sentence-level representation, you choose a pooling strategy. **[CLS] pooling** takes only the first token vector — during pre-training, [CLS] was designed to accumulate sentence-level context. **Mean pooling** averages all token vectors (excluding padding). In practice, mean pooling tends to produce better sentence embeddings because it incorporates all tokens rather than relying on one. For production-quality sentence similarity tasks, neither is ideal — that is what **SBERT** (Phase 3) fixes by fine-tuning on sentence pairs so [CLS] embeddings become geometrically meaningful.

**-> Phase 3: SBERT fine-tunes BERT specifically for sentence similarity, making semantic search practical.**
